# Room Temperature Forecast: Timestral vs Baseline

This notebook forecasts hourly room temperature with **Mistral Timestral**, a pretrained time-series model, and checks whether it beats a simple baseline.

**How it works**
1. Load `room_temperature.csv` (hourly readings, column `target` in deg C).
2. Hold back the last 7 days as a test set.
3. Forecast those 7 days with Timestral. This is *zero-shot*: the model is not trained on this data.
4. Compare it with a baseline that repeats the previous week.
5. Plot both against the real readings.

**Before you start:** Python 3.10+, Git installed, and `room_temperature.csv` in the same folder as this notebook.

## 1. Install packages
Run this cell once. Afterwards, restart the kernel so the new packages are picked up.

In [ ]:
# Run once, then restart the kernel (Kernel -> Restart) before running the next cells.
#!pip install -q numpy pandas matplotlib torch git+https://github.com/mistralai/timestral.git

## 2. Imports and settings
Change `MODEL_ID` for a different model size (bigger is more accurate but slower), or `H` for a different forecast length.

In [ ]:
import numpy as np                    # arrays and error metrics
import pandas as pd                   # loading the CSV and handling the time index
import torch                          # runs the Timestral model
import matplotlib.pyplot as plt       # plotting
from timestral import TimestralForecaster

# ---- Settings you may want to change ----
DATA_PATH = "room_temperature.csv"    # CSV must be in the same folder as this notebook
MODEL_ID  = "mistralai/Timestral-28M" # options: Timestral-5M / 28M / 123M / 209M
H         = 24 * 7                    # forecast horizon: 168 hours = 7 days

## 3. Load and split the data
The series is put on a strict hourly grid, then split into **train** (everything except the last 7 days) and **test** (the last 7 days).

In [ ]:
# Load only the columns we need: the timestamp and the temperature ("target").
# The "id" and "Datetime1" columns are just counters and are not used.
y = (pd.read_csv(DATA_PATH, parse_dates=["datetime"])
       .set_index("datetime")["target"]
       .sort_index()          # make sure readings are oldest -> newest
       .asfreq("h")           # force one row per hour; any missing hour becomes NaN
       .astype("float32"))

y.head()

In [ ]:
# Hold back the last 7 days: the model never sees them, so we can score the forecast.
train, test = y.iloc[:-H], y.iloc[-H:]
print(f"Missing hours: {int(y.isna().sum())}")
print(f"Train: {len(train)} hours | Test: {len(test)} hours")

## 4. Load the model
The weights download from Hugging Face the first time and are cached after that.

In [ ]:
# Load the pretrained model from Hugging Face (downloaded on first run, then cached).
# Uses the GPU if one is available, otherwise the CPU (fine for the 28M model).
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TimestralForecaster.from_pretrained(MODEL_ID, device=device)
print(f"Loaded {MODEL_ID} on {device}")

## 5. Forecast and baseline
Timestral gives a range of possible values, not just one. We keep the median (best guess) and the 10th/90th percentiles (80% range).

The **baseline** is a "seasonal naive" forecast: each hour next week equals the same hour last week. A useful model should beat it.

In [ ]:
# Zero-shot forecast: the model only sees the training history, no fitting needed.
fc = model.forecast(train.to_numpy(), prediction_length=H)

# The model returns arrays shaped (batch, variates, horizon) = (1, 1, 168).
# .squeeze() flattens them to (168,) so they line up with the test timestamps.
pred = np.asarray(fc.median).squeeze()         # most likely value (50th percentile)
lo   = np.asarray(fc.quantile(0.1)).squeeze()  # 10th percentile
hi   = np.asarray(fc.quantile(0.9)).squeeze()  # 90th percentile -> lo..hi is an 80% range

# Baseline to beat: assume the next 7 days repeat the previous 7 days, hour by hour.
naive = train.iloc[-H:].to_numpy()

## 6. Compare accuracy
- **MAE**: average size of the error in deg C. Lower is better.
- **RMSE**: like MAE but penalises big misses more. Lower is better.
- **Coverage**: how often the real reading falls in the 80% range. Around 80% is ideal.

In [ ]:
act = test.to_numpy()

def mae(p):  return np.mean(np.abs(p - act))           # average error in deg C
def rmse(p): return np.sqrt(np.mean((p - act) ** 2))   # punishes large misses more

results = pd.DataFrame({
    "MAE (deg C)":  [mae(pred),  mae(naive)],
    "RMSE (deg C)": [rmse(pred), rmse(naive)],
}, index=["Timestral", "Baseline (last week)"]).round(3)
display(results)

gain = (1 - mae(pred) / mae(naive)) * 100
print(f"Timestral MAE is {gain:.1f}% {'lower' if gain > 0 else 'higher'} than the baseline.")

# Coverage: share of actual readings inside the 80% range. Close to 80% = well calibrated.
coverage = np.mean((act >= lo) & (act <= hi)) * 100
print(f"80% range coverage: {coverage:.1f}%")

## 7. Plot
Orange line close to the black dashed line = good forecast. The shaded band shows the model's uncertainty.

In [ ]:
# Show the last 2 weeks of history, then the test week with both forecasts.
hist = train.iloc[-24 * 14:]

plt.figure(figsize=(13, 4))
plt.plot(hist.index, hist, color="steelblue", label="history")
plt.plot(test.index, act, "k--", label="actual")
plt.plot(test.index, naive, color="gray", alpha=.7, label="baseline (last week)")
plt.plot(test.index, pred, color="darkorange", label="Timestral (median)")
plt.fill_between(test.index, lo, hi, color="darkorange", alpha=.2, label="80% range")
plt.ylabel("deg C")
plt.title("Room temperature: Timestral vs baseline, last 7 days")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Troubleshooting
- **`ModuleNotFoundError: timestral`**: restart the kernel after installing; make sure Git is installed.
- **`x and y must have same first dimension`**: the forecast was not flattened; keep the `.squeeze()` calls in step 5.
- **Slow on CPU**: use `Timestral-5M` or `Timestral-28M`.